# Advanced Models: Random Forest and XGBoost with Hyperparameter Tuning

This notebook implements Random Forest and XGBoost regressors with:
- K-Fold Cross Validation
- Hyperparameter Tuning using GridSearchCV
- Model Evaluation and Comparison

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Model imports
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Cross-validation and hyperparameter tuning
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

In [2]:
# Load the data from the data/final directory
X_train = pd.read_csv('../data/final/X_train.csv')
X_test = pd.read_csv('../data/final/X_test.csv')
y_train = pd.read_csv('../data/final/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/final/y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (2231, 11)
X_test shape: (558, 11)
y_train shape: (2231,)
y_test shape: (558,)


## 1. Random Forest Regressor with K-Fold CV and Hyperparameter Tuning

Random Forest is an ensemble method that builds multiple decision trees and combines their predictions.

In [3]:
# Step 1: Define hyperparameter grid for Random Forest
# These parameters control the behavior of the Random Forest
rf_param_grid = {
    'n_estimators': [100, 200, 300],          # Number of trees in the forest
    'max_depth': [10, 20, 30, None],          # Maximum depth of trees (None = unlimited)
    'min_samples_split': [2, 5, 10],          # Minimum samples required to split a node
    'min_samples_leaf': [1, 2, 4],            # Minimum samples required at leaf node
    'max_features': ['sqrt', 'log2']          # Number of features to consider for splits
}

# Step 2: Create base Random Forest Regressor
rf_model = RandomForestRegressor(
    random_state=42,
    verbose=0
)

# Step 3: Set up K-Fold Cross Validation
kfold = KFold(
    n_splits=5,                  
    shuffle=True,                
    random_state=42
)

print("Random Forest Hyperparameter Tuning - GridSearchCV")

rf_grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_param_grid,
    cv=kfold,                      # Use k-fold cross validation
    scoring='neg_mean_squared_error',  # Scoring metric
    n_jobs=-1,                     # Parallel computation
    verbose=1
)

# Fit the grid search
rf_grid_search.fit(X_train, y_train)

# Step 5: Display best parameters and score
print("Random Forest - Best Results")
print(f"Best Parameters: {rf_grid_search.best_params_}")
print(f"Best CV Score (MSE): {-rf_grid_search.best_score_:.4f}")
print(f"Best Model: {rf_grid_search.best_estimator_}")

# Step 6: Get the best model
best_rf_model = rf_grid_search.best_estimator_

Random Forest Hyperparameter Tuning - GridSearchCV
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Random Forest - Best Results
Best Parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Best CV Score (MSE): 697202.8064
Best Model: RandomForestRegressor(max_depth=10, max_features='sqrt', n_estimators=300,
                      random_state=42)


In [ ]:
# Step 7: Evaluate Random Forest on training and test sets
y_train_pred_rf = best_rf_model.predict(X_train)
y_test_pred_rf = best_rf_model.predict(X_test)

# Calculate performance metrics
rf_train_r2 = r2_score(y_train, y_train_pred_rf)
rf_test_r2 = r2_score(y_test, y_test_pred_rf)
rf_train_mse = mean_squared_error(y_train, y_train_pred_rf)
rf_test_mse = mean_squared_error(y_test, y_test_pred_rf)
rf_train_mae = mean_absolute_error(y_train, y_train_pred_rf)
rf_test_mae = mean_absolute_error(y_test, y_test_pred_rf)
rf_train_rmse = np.sqrt(rf_train_mse)
rf_test_rmse = np.sqrt(rf_test_mse)

# Display metrics
print("Random Forest - Performance Metrics")
print(f"\nTraining Set:")
print(f"  R² Score: {rf_train_r2:.4f}")
print(f"  RMSE: {rf_train_rmse:.4f}")
print(f"  MSE: {rf_train_mse:.4f}")
print(f"  MAE: {rf_train_mae:.4f}")

print(f"\nTest Set:")
print(f"  R² Score: {rf_test_r2:.4f}")
print(f"  RMSE: {rf_test_rmse:.4f}")
print(f"  MSE: {rf_test_mse:.4f}")
print(f"  MAE: {rf_test_mae:.4f}")

# Feature importance
feature_importance_rf = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "=" * 60)
print("Top 10 Important Features - Random Forest")
print("=" * 60)
print(feature_importance_rf.head(10))


Random Forest - Performance Metrics

Training Set:
  R² Score: 0.9919
  RMSE: 832.8802
  MSE: 693689.3808
  MAE: 580.6327

Test Set:
  R² Score: 0.9911
  RMSE: 850.4634
  MSE: 723287.9858
  MAE: 593.3191

Top 10 Important Features - Random Forest
                      Feature  Importance
9              Potassh_50Days    0.118950
8       Micronutrients_70Days    0.104559
2                   Hectares     0.102749
7                 Urea_40Days    0.096862
4   LP_nurseryarea(in Tonnes)    0.093996
10                 DAP_20days    0.089503
5             Seedrate(in Kg)    0.085753
0         Weed28D_thiobencarb    0.083134
3     LP_Mainfield(in Tonnes)    0.079998
1        Nursery area (Cents)    0.079918


In [6]:
# Save the tuned Random Forest model to a .pkl file (for later reuse)
import joblib
from pathlib import Path

model_dir = Path('../models')
model_dir.mkdir(parents=True, exist_ok=True)

rf_model_path = model_dir / 'random_forest_best.pkl'
joblib.dump(best_rf_model, rf_model_path)

print(f"Random Forest model saved to: {rf_model_path}")

Random Forest model saved to: ..\models\random_forest_best.pkl
